# Занятие 4. Оптимизация, Дистилляция и Mutual Learning

## Цели занятия:
- Изучить влияние оптимизаторов и планировщиков скорости обучения
- Реализовать дистилляцию знаний (Knowledge Distillation)
- Реализовать взаимное обучение (Deep Mutual Learning)
- Сравнить эффективность сжатия моделей

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import numpy as np
import random

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

---
## Шаг 1. Подготовка данных и моделей

In [ ]:
# Data preparation for CIFAR-10
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=0)

print(f"Train samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

In [ ]:
# Teacher Model (ResNet18 adapted for CIFAR-10)
class TeacherNet(nn.Module):
    """Teacher model based on ResNet18 for CIFAR-10."""
    
    def __init__(self):
        super(TeacherNet, self).__init__()
        self.resnet = models.resnet18(weights=None)
        # Adapt first conv for 32x32 images
        self.resnet.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.resnet.maxpool = nn.Identity()  # Remove maxpool for small images
        self.resnet.fc = nn.Linear(512, 10)
    
    def forward(self, x):
        return self.resnet(x)

# Student Model (Small CNN)
class StudentCNN(nn.Module):
    """Small student model for knowledge distillation."""
    
    def __init__(self):
        super(StudentCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv3 = nn.Conv2d(64, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 4 * 4, 128)
        self.fc2 = nn.Linear(128, 10)
        self.dropout = nn.Dropout(0.25)
    
    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))  # 32x32 -> 16x16
        x = self.pool(F.relu(self.conv2(x)))  # 16x16 -> 8x8
        x = self.pool(F.relu(self.conv3(x)))  # 8x8 -> 4x4
        x = x.view(-1, 64 * 4 * 4)
        x = self.dropout(F.relu(self.fc1(x)))
        return self.fc2(x)

# Count parameters
teacher = TeacherNet().to(device)
student = StudentCNN().to(device)

print(f"Teacher parameters: {sum(p.numel() for p in teacher.parameters()):,}")
print(f"Student parameters: {sum(p.numel() for p in student.parameters()):,}")

---
## Шаг 2. Обучение Teacher модели

In [ ]:
def evaluate(model, loader, device):
    """Evaluate model accuracy."""
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100 * correct / total

def train_teacher(model, train_loader, test_loader, epochs=5):
    """Train teacher model."""
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.5)
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
        
        scheduler.step()
        acc = evaluate(model, test_loader, device)
        print(f"Epoch {epoch+1}: Loss={running_loss/len(train_loader):.4f}, Acc={acc:.2f}%")
    
    return model

In [ ]:
print("="*50)
print("Training Teacher Model")
print("="*50)
set_seed(42)
teacher = TeacherNet().to(device)
teacher = train_teacher(teacher, train_loader, test_loader, epochs=5)

---
## Шаг 3. Реализация Distillation Loss

In [ ]:
def distillation_loss(student_outputs, teacher_outputs, labels, T=4.0, alpha=0.7):
    """Knowledge Distillation loss.
    
    Args:
        student_outputs: Raw logits from student
        teacher_outputs: Raw logits from teacher (frozen)
        labels: Ground truth labels
        T: Temperature for softening distributions
        alpha: Weight for hard loss (CE with labels)
    """
    # Hard loss: Cross Entropy with ground truth
    ce_loss = nn.CrossEntropyLoss()(student_outputs, labels)
    
    # Soft loss: KL Divergence with teacher
    # IMPORTANT: Use log_softmax for student and softmax for teacher
    kd_loss = nn.KLDivLoss(reduction='batchmean')(
        F.log_softmax(student_outputs / T, dim=1),
        F.softmax(teacher_outputs / T, dim=1)
    ) * (T * T)  # Scale by T^2
    
    return alpha * ce_loss + (1 - alpha) * kd_loss

In [ ]:
def train_with_distillation(student, teacher, train_loader, test_loader, epochs=5, T=4.0, alpha=0.7):
    """Train student with Knowledge Distillation."""
    teacher.eval()  # Teacher is frozen
    
    optimizer = torch.optim.Adam(student.parameters(), lr=0.001)
    history = {'train_loss': [], 'val_acc': []}
    
    for epoch in range(epochs):
        student.train()
        running_loss = 0.0
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            student_out = student(images)
            
            with torch.no_grad():
                teacher_out = teacher(images)
            
            loss = distillation_loss(student_out, teacher_out, labels, T=T, alpha=alpha)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
        
        acc = evaluate(student, test_loader, device)
        history['train_loss'].append(running_loss / len(train_loader))
        history['val_acc'].append(acc)
        print(f"Epoch {epoch+1}: Loss={running_loss/len(train_loader):.4f}, Acc={acc:.2f}%")
    
    return history

In [ ]:
# Train student with KD
print("="*50)
print("Training Student with Knowledge Distillation")
print("="*50)
set_seed(42)
student_kd = StudentCNN().to(device)
history_kd = train_with_distillation(student_kd, teacher, train_loader, test_loader, epochs=5)

---
## Шаг 4. Реализация Mutual Learning

In [ ]:
def mutual_learning_loss(out_A, out_B, labels, T=4.0, lambda_kl=0.5):
    """Deep Mutual Learning loss.
    
    Two students learn from each other using KL divergence.
    """
    # Individual CE losses
    ce_A = nn.CrossEntropyLoss()(out_A, labels)
    ce_B = nn.CrossEntropyLoss()(out_B, labels)
    
    # Peer knowledge exchange (both directions)
    kl_A = nn.KLDivLoss(reduction='batchmean')(
        F.log_softmax(out_A / T, dim=1),
        F.softmax(out_B / T, dim=1)
    ) * (T * T)
    
    kl_B = nn.KLDivLoss(reduction='batchmean')(
        F.log_softmax(out_B / T, dim=1),
        F.softmax(out_A / T, dim=1)
    ) * (T * T)
    
    loss_A = ce_A + lambda_kl * kl_A
    loss_B = ce_B + lambda_kl * kl_B
    
    return loss_A, loss_B

In [ ]:
def train_mutual_learning(student_A, student_B, train_loader, test_loader, epochs=5):
    """Train two students with Mutual Learning."""
    opt_A = torch.optim.Adam(student_A.parameters(), lr=0.001)
    opt_B = torch.optim.Adam(student_B.parameters(), lr=0.001)
    
    history = {'train_loss': [], 'val_acc_A': [], 'val_acc_B': []}
    
    for epoch in range(epochs):
        student_A.train()
        student_B.train()
        running_loss = 0.0
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            
            opt_A.zero_grad()
            opt_B.zero_grad()
            
            out_A = student_A(images)
            out_B = student_B(images)
            
            loss_A, loss_B = mutual_learning_loss(out_A, out_B, labels)
            
            loss_A.backward()
            loss_B.backward()
            
            opt_A.step()
            opt_B.step()
            
            running_loss += (loss_A.item() + loss_B.item()) / 2
        
        acc_A = evaluate(student_A, test_loader, device)
        acc_B = evaluate(student_B, test_loader, device)
        
        history['train_loss'].append(running_loss / len(train_loader))
        history['val_acc_A'].append(acc_A)
        history['val_acc_B'].append(acc_B)
        
        print(f"Epoch {epoch+1}: Loss={running_loss/len(train_loader):.4f}, "
              f"Acc_A={acc_A:.2f}%, Acc_B={acc_B:.2f}%")
    
    return history

In [ ]:
# Train with Mutual Learning
print("="*50)
print("Training with Mutual Learning")
print("="*50)
set_seed(42)
student_A = StudentCNN().to(device)
student_B = StudentCNN().to(device)
history_ml = train_mutual_learning(student_A, student_B, train_loader, test_loader, epochs=5)

---
## Шаг 5. Сравнение результатов

In [ ]:
# Baseline: Train student without distillation
print("="*50)
print("Training Student Baseline (no distillation)")
print("="*50)
set_seed(42)
student_baseline = StudentCNN().to(device)

optimizer = torch.optim.Adam(student_baseline.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

for epoch in range(5):
    student_baseline.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = student_baseline(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    acc = evaluate(student_baseline, test_loader, device)
    print(f"Epoch {epoch+1}: Loss={running_loss/len(train_loader):.4f}, Acc={acc:.2f}%")

baseline_acc = evaluate(student_baseline, test_loader, device)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Summary
results = {
    'Method': ['Student Baseline', 'Student + KD', 'Student A (DML)', 'Student B (DML)'],
    'Accuracy (%)': [
        baseline_acc,
        history_kd['val_acc'][-1],
        history_ml['val_acc_A'][-1],
        history_ml['val_acc_B'][-1]
    ],
    'Params': [
        sum(p.numel() for p in student_baseline.parameters()),
        sum(p.numel() for p in student_kd.parameters()),
        sum(p.numel() for p in student_A.parameters()),
        sum(p.numel() for p in student_B.parameters())
    ]
}

df = pd.DataFrame(results)
print("\n" + "="*50)
print("RESULTS COMPARISON")
print("="*50)
print(df.to_string(index=False))

In [ ]:
# Visualization
fig, ax = plt.subplots(figsize=(10, 6))

methods = results['Method']
accs = results['Accuracy (%)']
colors = ['gray', 'steelblue', 'coral', 'coral']

bars = ax.bar(methods, accs, color=colors, edgecolor='black')
ax.set_ylabel('Test Accuracy (%)', fontsize=12)
ax.set_title('Knowledge Distillation vs Mutual Learning', fontsize=14)
ax.grid(True, alpha=0.3, axis='y')

for bar, acc in zip(bars, accs):
    ax.annotate(f'{acc:.1f}%', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 3), textcoords="offset points", ha='center', fontsize=11)

plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig('distillation_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Домашнее задание

1. Подобрать оптимальную температуру T для дистилляции (2, 4, 8)
2. Сравнить SGD и Adam оптимизаторы для Student модели
3. Сохранить веса лучшей модели Student

---
## Контрольные вопросы

1. Зачем нужно делить логиты на температуру T?
2. В чем преимущество DML перед классической KD?
3. Почему Teacher модель должна быть заморожена?
4. Как влияет коэффициент alpha на баланс потерь?